# Module 2 — automatic anchor selection (rule a′, reviewer 2026-10-17/18)

Phase-1 runtime (L4 is enough). Runs on the SEVEN existing pairs only: the census clips are the clean test and are never used here.

Order: 0 mount + repo · 1 bootstrap (D4RT, SAM2) · 2 extra deps (MoGe, pyiqa) · 3 measure the candidates (GPU) · 4 select (CPU) · 5 copy the reports back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/repo
!git pull

In [ ]:
!bash scripts/bootstrap.sh

In [ ]:
%%bash
pip uninstall -y -q torchaudio
pip install -q --no-deps pyiqa
pip install -q git+https://github.com/microsoft/MoGe.git
python -c "import torch, sam2, pyiqa, moge; print('deps ok', torch.__version__, torch.cuda.is_available())"

## 3. Measure every candidate (GPU, ~1 min per candidate)
Per candidate image: the subject mask (OWLv2 + SAM2, the instrument of record), proxy IoU, silhouette margin, CLIP identity, MUSIQ; G1 = MoGe depth → ground band determinability + contact against the proxy; G2 = path support (reported; gated once `tau_sup` exists). Resumable: a measured candidate is skipped.

In [ ]:
%%bash
python scripts/select_m2d_anchors.py measure --pairs 000000000035.1.003/vehicle 000000000265.2.003/vehicle 000000000265.2.005/vehicle 000000000265.2.005/human 000000000284.0.006/vehicle 000000000370.1.008/human 000000000404.1.004/vehicle --force

In [ ]:
%%bash
P="000000000035.1.003/vehicle 000000000265.2.003/vehicle 000000000265.2.005/vehicle 000000000265.2.005/human 000000000284.0.006/vehicle 000000000370.1.008/human 000000000404.1.004/vehicle"
python scripts/select_m2d_anchors.py measure --force --pairs $P
python scripts/select_m2d_anchors.py selftest --pairs $P
python scripts/select_m2d_anchors.py select --pairs $P --write-anchor


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/select_m2d_anchors.py g2selftest --clips $(python -c "import json; print(' '.join(r['clip'] for r in json.load(open('measurements/census/census.json')) if r['census']=='PASS'))")


In [ ]:
!MT_ROOT=/content/drive/MyDrive/motion_transfer_exp python scripts/select_m2d_anchors.py g2selftest --clips $(python -c "import json; print(' '.join(r['clip'] for r in json.load(open('measurements/census/census.json')) if r['census']=='PASS'))")


In [ ]:
!python scripts/select_m2d_anchors.py selftest --pairs 000000000035.1.003/vehicle 000000000265.2.003/vehicle 000000000265.2.005/vehicle 000000000265.2.005/human 000000000284.0.006/vehicle 000000000370.1.008/human 000000000404.1.004/vehicle


## 4. Select (CPU, seconds)
Applies the rule with the lock's calibrated values; writes `selection_v2/selection.json` per pair and `reports/m2_selection/selection.{json,md}` with the acceptable-pick rate against the blind verdicts, the agreement with the human's picks (descriptive), and the ε sensitivity.

In [ ]:
%%bash
python scripts/select_m2d_anchors.py select --pairs 000000000035.1.003/vehicle 000000000265.2.003/vehicle 000000000265.2.005/vehicle 000000000265.2.005/human 000000000284.0.006/vehicle 000000000370.1.008/human 000000000404.1.004/vehicle
cat /content/drive/MyDrive/motion_transfer/reports/m2_selection/selection.md

## 5. Copy back
Copy `motion_transfer/reports/m2_selection/` into the repo's `measurements/m2_selection/`, and each pair's `cache/m2_d/<clip>/<cat>/selection_v2/` into `measurements/m2_selection/<clip>_<cat>/` (two small json files per pair).